# Lab 10 – Notebook 01: Unsupervised Learning and Clustering

### In-class demonstration

This notebook develops clustering as a sequence of ideas:

**unlabeled data → K-Means by hand → K-Means with Scikit-learn → cluster quality → categorical/mixed data → limitations of K-Means → DBSCAN**

The aim is to connect the mathematics in the lecture to the code.

## 1. Setup

We will use NumPy, pandas, Matplotlib, and Scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.set_printoptions(precision=3, suppress=True)

## 2. Supervised vs unsupervised learning

In supervised learning, the training examples include labels.

In clustering, the input contains only the feature vectors:

\[
X = \{x^{(1)}, x^{(2)}, \ldots, x^{(m)}\}
\]

The goal is to discover structure or groups from the data itself.

In [ ]:
X_demo, _ = make_blobs(
    n_samples=80,
    centers=3,
    cluster_std=0.8,
    random_state=RANDOM_STATE
)

plt.figure(figsize=(6, 4))
plt.scatter(X_demo[:, 0], X_demo[:, 1])
plt.title("Unlabeled data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# Part A – K-Means mathematically

K-Means alternates between two steps:

1. **Assignment:** assign each point to the nearest centroid.
2. **Update:** move each centroid to the mean of the points assigned to it.

The process repeats until the assignments or centroids stop changing substantially.

## 3. A small dataset we can solve by hand

We deliberately use only six points so that the mathematics is visible.

In [ ]:
X_small = np.array([
    [1.0, 1.0],
    [1.5, 2.0],
    [2.0, 1.0],
    [5.0, 5.0],
    [6.0, 5.0],
    [5.5, 6.0]
])

initial_centroids = np.array([
    [1.0, 4.0],
    [6.0, 2.0]
])

plt.figure(figsize=(6, 4))
plt.scatter(X_small[:, 0], X_small[:, 1], s=70)
plt.scatter(
    initial_centroids[:, 0],
    initial_centroids[:, 1],
    marker="X",
    s=180
)
for i, (x, y) in enumerate(X_small):
    plt.text(x + 0.08, y + 0.08, f"P{i+1}")
plt.title("Six points and two initial centroids")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

## 4. Assignment step

For each point \(x^{(i)}\), K-Means chooses the nearest centroid:

\[
c^{(i)}
=
\arg\min_k
\left\|x^{(i)} - \mu_k\right\|^2
\]

The code below calculates every point-to-centroid distance.

In [ ]:
def distance_matrix(X, centroids):
    return np.linalg.norm(
        X[:, None, :] - centroids[None, :, :],
        axis=2
    )

distances_0 = distance_matrix(X_small, initial_centroids)
labels_0 = np.argmin(distances_0, axis=1)

distance_table = pd.DataFrame(
    distances_0,
    columns=["Distance to C1", "Distance to C2"]
)
distance_table["Assigned cluster"] = labels_0
distance_table

### What happened?

`np.argmin(..., axis=1)` returns the column containing the smallest distance for each row.

That is exactly the mathematical **arg min** in the K-Means assignment equation.

## 5. Update step

After assigning points, the new centroid of cluster \(k\) is the mean of all points in that cluster:

\[
\mu_k
=
\frac{1}{|C_k|}
\sum_{x^{(i)}\in C_k}
x^{(i)}
\]

In [ ]:
def update_centroids(X, labels, K):
    return np.array([
        X[labels == k].mean(axis=0)
        for k in range(K)
    ])

centroids_1 = update_centroids(X_small, labels_0, K=2)

print("Initial centroids:")
print(initial_centroids)

print("\nUpdated centroids after one assignment step:")
print(centroids_1)

In [ ]:
plt.figure(figsize=(6, 4))
for k in range(2):
    pts = X_small[labels_0 == k]
    plt.scatter(pts[:, 0], pts[:, 1], s=70, label=f"Cluster {k}")

plt.scatter(
    initial_centroids[:, 0],
    initial_centroids[:, 1],
    marker="x",
    s=130,
    label="Old centroids"
)
plt.scatter(
    centroids_1[:, 0],
    centroids_1[:, 1],
    marker="X",
    s=180,
    label="Updated centroids"
)

plt.title("K-Means after one update")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

## 6. Repeat the assignment and update

K-Means continues the same two operations until the centroids stabilize.

In [ ]:
centroids = initial_centroids.copy()

history = []

for iteration in range(10):
    distances = distance_matrix(X_small, centroids)
    labels = np.argmin(distances, axis=1)
    new_centroids = update_centroids(X_small, labels, K=2)

    shift = np.linalg.norm(new_centroids - centroids)
    history.append((iteration + 1, centroids.copy(), labels.copy(), shift))

    centroids = new_centroids

    if shift < 1e-8:
        break

for iteration, old_centroids, labels, shift in history:
    print(
        f"Iteration {iteration}: "
        f"centroid shift = {shift:.4f}, "
        f"assignments = {labels}"
    )

print("\nFinal centroids:")
print(centroids)

## 7. K-Means cost function / inertia

K-Means minimizes the total squared distance from each point to the centroid of its assigned cluster:

\[
J
=
\sum_{k=1}^{K}
\sum_{x^{(i)}\in C_k}
\left\|x^{(i)} - \mu_k\right\|^2
\]

Scikit-learn calls this quantity **inertia**.

Smaller inertia means the points are more compact around their own centroids.

In [ ]:
def manual_inertia(X, labels, centroids):
    total = 0.0

    for k in range(len(centroids)):
        cluster_points = X[labels == k]
        squared_distances = np.sum(
            (cluster_points - centroids[k]) ** 2,
            axis=1
        )
        total += squared_distances.sum()

    return total

final_distances = distance_matrix(X_small, centroids)
final_labels = np.argmin(final_distances, axis=1)

J_manual = manual_inertia(X_small, final_labels, centroids)

print(f"Manual K-Means cost / inertia = {J_manual:.4f}")

### Compare our mathematics with Scikit-learn

We now fit Scikit-learn to the same six points.

Because cluster labels can be swapped (`0` and `1` have no semantic meaning), compare the **centroids and inertia**, not the numeric cluster names.

In [ ]:
sk_small = KMeans(
    n_clusters=2,
    init=initial_centroids,
    n_init=1,
    random_state=RANDOM_STATE
)

sk_small.fit(X_small)

print("Scikit-learn centroids:")
print(sk_small.cluster_centers_)

print(f"Scikit-learn inertia = {sk_small.inertia_:.4f}")
print(f"Manual inertia       = {J_manual:.4f}")

# Part B – K-Means in practice

## 8. Apply K-Means to a larger synthetic dataset

In [ ]:
X_blobs, _ = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=[0.8, 1.0, 0.7],
    random_state=RANDOM_STATE
)

kmeans = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=RANDOM_STATE
)

blob_labels = kmeans.fit_predict(X_blobs)

print("Centroids:")
print(kmeans.cluster_centers_)

print(f"\nInertia: {kmeans.inertia_:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))

for k in range(3):
    pts = X_blobs[blob_labels == k]
    plt.scatter(pts[:, 0], pts[:, 1], s=25, label=f"Cluster {k}")

plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X",
    s=180,
    label="Centroids"
)

plt.title("K-Means clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 9. Random initialization and local optima

K-Means can converge to different local solutions depending on the starting centroids.

We run K-Means several times with:

- `init='random'`
- `n_init=1`

and compare the resulting inertia values.

Then we compare that with a model that uses several initializations.

In [ ]:
initialization_results = []

for seed in range(8):
    model = KMeans(
        n_clusters=3,
        init="random",
        n_init=1,
        random_state=seed
    )
    model.fit(X_blobs)

    initialization_results.append({
        "random_state": seed,
        "inertia": model.inertia_
    })

init_df = pd.DataFrame(initialization_results)
init_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(
    init_df["random_state"],
    init_df["inertia"],
    marker="o"
)
plt.xlabel("Random initialization")
plt.ylabel("Final inertia")
plt.title("Different initializations can give different costs")
plt.show()

best_single = init_df["inertia"].min()

multi_start = KMeans(
    n_clusters=3,
    init="random",
    n_init=20,
    random_state=RANDOM_STATE
)
multi_start.fit(X_blobs)

print(f"Best inertia among shown single runs: {best_single:.3f}")
print(f"Inertia using n_init=20:             {multi_start.inertia_:.3f}")

### Teaching point

`n_init` asks K-Means to try several initial centroid configurations and retain the solution with the lowest inertia.

This reduces the risk of accepting a poor local optimum.

# Part C – Choosing the number of clusters

## 10. Elbow Method

For each candidate \(K\), fit K-Means and record inertia.

Inertia always decreases as \(K\) increases, so we look for the point where the improvement starts becoming much smaller.

In [ ]:
K_values = range(1, 9)
inertias = []

for K in K_values:
    model = KMeans(
        n_clusters=K,
        n_init=10,
        random_state=RANDOM_STATE
    )
    model.fit(X_blobs)
    inertias.append(model.inertia_)

elbow_df = pd.DataFrame({
    "K": list(K_values),
    "Inertia": inertias
})
elbow_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(list(K_values), inertias, marker="o")
plt.xlabel("Number of clusters, K")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

## 11. Silhouette Coefficient

Inertia measures only **within-cluster compactness**.

Silhouette considers both:

- **cohesion**: how close a point is to its own cluster;
- **separation**: how far it is from the nearest competing cluster.

For point \(i\):

\[
a(i)
=
\text{average distance from } i
\text{ to points in its own cluster}
\]

\[
b(i)
=
\text{smallest average distance from } i
\text{ to any other cluster}
\]

\[
s(i)
=
\frac{b(i)-a(i)}
{\max(a(i),b(i))}
\]

A value close to **1** is good, around **0** suggests overlap, and a negative value suggests a potentially poor assignment.

In [ ]:
silhouette_results = []

for K in range(2, 9):
    model = KMeans(
        n_clusters=K,
        n_init=10,
        random_state=RANDOM_STATE
    )
    labels = model.fit_predict(X_blobs)

    score = silhouette_score(X_blobs, labels)

    silhouette_results.append({
        "K": K,
        "Silhouette": score
    })

silhouette_df = pd.DataFrame(silhouette_results)
silhouette_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(
    silhouette_df["K"],
    silhouette_df["Silhouette"],
    marker="o"
)
plt.xlabel("Number of clusters, K")
plt.ylabel("Silhouette score")
plt.title("Silhouette score for candidate K values")
plt.show()

## 12. Calculate one Silhouette value manually

To make the formula concrete, choose one observation from the \(K=3\) solution.

We will calculate:

- \(a(i)\): mean distance to other points in its own cluster;
- \(b(i)\): mean distance to each other cluster, then select the smallest;
- \(s(i)\): the final silhouette value.

In [ ]:
model_k3 = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=RANDOM_STATE
)
labels_k3 = model_k3.fit_predict(X_blobs)

i = 0
point = X_blobs[i]
own_cluster = labels_k3[i]

all_distances = np.linalg.norm(X_blobs - point, axis=1)

same_mask = labels_k3 == own_cluster
same_mask[i] = False

a_i = all_distances[same_mask].mean()

other_cluster_means = {}

for k in sorted(set(labels_k3)):
    if k == own_cluster:
        continue

    mean_distance = all_distances[labels_k3 == k].mean()
    other_cluster_means[k] = mean_distance

b_i = min(other_cluster_means.values())

s_i = (b_i - a_i) / max(a_i, b_i)

print("Selected point:", point)
print("Assigned cluster:", own_cluster)
print(f"a(i) = {a_i:.4f}")
print("Mean distance to each other cluster:", other_cluster_means)
print(f"b(i) = {b_i:.4f}")
print(f"Manual silhouette s(i) = {s_i:.4f}")

In [ ]:
from sklearn.metrics import silhouette_samples

sil_samples = silhouette_samples(X_blobs, labels_k3)

print(
    "Scikit-learn silhouette for the same point =",
    round(sil_samples[i], 4)
)

### Compactness vs separation

A good clustering solution should ideally have:

\[
\text{small within-cluster distances}
\]

and

\[
\text{large between-cluster distances}
\]

This is why Inertia and Silhouette tell us different but complementary things.

# Part D – Categorical and mixed data

## 13. K-Modes intuition

K-Means uses means and Euclidean distance, so it is designed for numerical data.

For categorical variables, K-Modes instead uses:

- **modes** as cluster representatives;
- **mismatch / Hamming-style distance**.

In [ ]:
categorical = pd.DataFrame({
    "Color": ["Red", "Blue", "Green", "Blue", "Red", "Green"],
    "Pet":   ["Dog", "Cat", "Dog", "Dog", "Cat", "Cat"],
    "Drink": ["Tea", "Coffee", "Tea", "Tea", "Coffee", "Tea"]
}, index=list("ABCDEF"))

categorical

In [ ]:
mode_0 = np.array(["Red", "Dog", "Tea"])
mode_1 = np.array(["Blue", "Cat", "Coffee"])

def categorical_mismatch_distance(row, mode):
    return np.sum(np.asarray(row) != np.asarray(mode))

rows = []

for idx, row in categorical.iterrows():
    d0 = categorical_mismatch_distance(row.values, mode_0)
    d1 = categorical_mismatch_distance(row.values, mode_1)

    rows.append({
        "Person": idx,
        "Distance to mode 0": d0,
        "Distance to mode 1": d1,
        "Assigned cluster": 0 if d0 <= d1 else 1
    })

kmodes_demo = pd.DataFrame(rows)
kmodes_demo

### Updating a mode

For each categorical feature, the new cluster representative is the most frequent category among the points currently assigned to that cluster.

In [ ]:
assigned = kmodes_demo.set_index("Person")["Assigned cluster"]

for k in [0, 1]:
    members = categorical.loc[assigned[assigned == k].index]
    print(f"Cluster {k} members:")
    display(members)

    print("Updated mode:")
    print(members.mode().iloc[0].to_dict())
    print()

## 14. K-Prototypes intuition

K-Prototypes is designed for **mixed numerical + categorical data**.

A typical combined distance is:

\[
D(x,q)
=
\sum_{j\in \text{numeric}}
(x_j-q_j)^2
+
\gamma
\sum_{j\in \text{categorical}}
\delta(x_j,q_j)
\]

where \(\gamma\) balances the numerical and categorical parts.

In [ ]:
def kprototype_distance(
    x_num, q_num,
    x_cat, q_cat,
    gamma=1.0
):
    numeric_part = np.sum((np.array(x_num) - np.array(q_num)) ** 2)
    categorical_part = np.sum(np.array(x_cat) != np.array(q_cat))

    return numeric_part + gamma * categorical_part

x_num = [30, 50]
q_num = [28, 48]

x_cat = ["Male", "Sales"]
q_cat = ["Male", "HR"]

print(
    "Combined distance =",
    kprototype_distance(
        x_num, q_num,
        x_cat, q_cat,
        gamma=2.0
    )
)

# Part E – When K-Means is not a good geometric match

## 15. Non-convex data

K-Means works best when clusters are compact and roughly spherical.

The two-moons dataset has curved clusters, so centroid-based partitions can cut through the natural shapes.

In [ ]:
X_moons, _ = make_moons(
    n_samples=300,
    noise=0.07,
    random_state=RANDOM_STATE
)

X_moons_scaled = StandardScaler().fit_transform(X_moons)

km_moons = KMeans(
    n_clusters=2,
    n_init=10,
    random_state=RANDOM_STATE
)

moon_km_labels = km_moons.fit_predict(X_moons_scaled)

plt.figure(figsize=(6, 4))

for label in np.unique(moon_km_labels):
    pts = X_moons_scaled[moon_km_labels == label]
    plt.scatter(pts[:, 0], pts[:, 1], s=20, label=f"Cluster {label}")

plt.scatter(
    km_moons.cluster_centers_[:, 0],
    km_moons.cluster_centers_[:, 1],
    marker="X",
    s=180,
    label="Centroids"
)

plt.title("K-Means on moon-shaped clusters")
plt.legend()
plt.show()

# Part F – DBSCAN

DBSCAN groups points using **density** rather than centroids.

Its two key parameters are:

- `eps`: neighbourhood radius;
- `min_samples`: minimum number of nearby points required for a dense/core region.

DBSCAN can also label observations as **noise**, represented by cluster label `-1`.

## 16. DBSCAN on the same moon-shaped data

In [ ]:
dbscan = DBSCAN(
    eps=0.23,
    min_samples=5
)

moon_db_labels = dbscan.fit_predict(X_moons_scaled)

print("Labels found:", np.unique(moon_db_labels))
print("Number of noise points:", np.sum(moon_db_labels == -1))

In [ ]:
plt.figure(figsize=(6, 4))

for label in np.unique(moon_db_labels):
    pts = X_moons_scaled[moon_db_labels == label]
    marker = "x" if label == -1 else "o"

    plt.scatter(
        pts[:, 0],
        pts[:, 1],
        s=25,
        marker=marker,
        label="Noise" if label == -1 else f"Cluster {label}"
    )

plt.title("DBSCAN on moon-shaped data")
plt.legend()
plt.show()

## 17. Core, border, and noise points in code

After fitting DBSCAN:

- core points are available through `core_sample_indices_`;
- non-core points that still belong to a cluster are border points;
- points with label `-1` are noise.

In [ ]:
core_mask = np.zeros(len(X_moons_scaled), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True

noise_mask = moon_db_labels == -1
border_mask = (~core_mask) & (~noise_mask)

print("Core points:  ", core_mask.sum())
print("Border points:", border_mask.sum())
print("Noise points: ", noise_mask.sum())

In [ ]:
plt.figure(figsize=(6, 4))

plt.scatter(
    X_moons_scaled[core_mask, 0],
    X_moons_scaled[core_mask, 1],
    s=28,
    marker="o",
    label="Core"
)

plt.scatter(
    X_moons_scaled[border_mask, 0],
    X_moons_scaled[border_mask, 1],
    s=45,
    marker="s",
    label="Border"
)

plt.scatter(
    X_moons_scaled[noise_mask, 0],
    X_moons_scaled[noise_mask, 1],
    s=50,
    marker="x",
    label="Noise"
)

plt.title("DBSCAN point types")
plt.legend()
plt.show()

## 18. Effect of changing `eps`

A small `eps` can leave many points disconnected and labelled as noise.

A large `eps` can merge groups that should remain separate.

In [ ]:
eps_results = []

for eps in [0.12, 0.18, 0.23, 0.35, 0.50]:
    model = DBSCAN(
        eps=eps,
        min_samples=5
    )
    labels = model.fit_predict(X_moons_scaled)

    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise = np.sum(labels == -1)

    eps_results.append({
        "eps": eps,
        "clusters": clusters,
        "noise_points": noise
    })

pd.DataFrame(eps_results)

## 19. Effect of changing `min_samples`

With the same `eps`, increasing `min_samples` makes the definition of a dense region stricter.

In [ ]:
min_samples_results = []

for min_samples in [3, 5, 8, 12, 20]:
    model = DBSCAN(
        eps=0.23,
        min_samples=min_samples
    )
    labels = model.fit_predict(X_moons_scaled)

    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise = np.sum(labels == -1)

    min_samples_results.append({
        "min_samples": min_samples,
        "clusters": clusters,
        "noise_points": noise
    })

pd.DataFrame(min_samples_results)

## 20. k-distance plot for choosing `eps`

A common heuristic is to inspect the distance from each point to its \(k\)-th nearest neighbour.

For example, if `min_samples = 5`, examine the 5-nearest-neighbour distance.

A noticeable bend in the sorted distance curve can provide a starting value for `eps`.

This is a heuristic rather than a guaranteed optimum.

In [ ]:
k = 5

nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_moons_scaled)

distances, indices = nn.kneighbors(X_moons_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(6, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by k-distance")
plt.ylabel(f"Distance to {k}th nearest neighbour")
plt.title("k-distance plot")
plt.show()

# Part G – Final comparison

## 21. K-Means vs DBSCAN

| Property | K-Means | DBSCAN |
|---|---|---|
| Main idea | Distance to centroids | Density connectivity |
| Need number of clusters beforehand? | Yes | No |
| Cluster shape | Best for compact / roughly spherical groups | Can recover irregular shapes |
| Explicit noise label | No | Yes |
| Important parameters | `K`, initialization | `eps`, `min_samples` |
| Typical quality tools | Inertia, Silhouette | Silhouette for clustered points, noise rate, domain interpretation |

The important lesson is not that one algorithm is always better. The clustering method must match the **data type, geometry, density structure, and purpose of the analysis**.

## 22. End-of-demonstration discussion

1. Why does K-Means alternate between an assignment step and a centroid-update step?
2. What exactly does `inertia_` measure?
3. Why can inertia not be used alone to choose \(K\)?
4. How does Silhouette add information that inertia does not?
5. Why can different K-Means initializations produce different results?
6. Why does DBSCAN work better than K-Means on the moon-shaped example?
7. What happens when `eps` or `min_samples` becomes too strict or too loose?